In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

RAW = Path("../data/raw")
plt.rcParams.update({"figure.figsize": (10, 4), "axes.grid": True, "grid.alpha": .3,
                     "axes.spines.top": False, "axes.spines.right": False})
pd.options.display.float_format = "{:,.2f}".format

In [ ]:
departments = pd.read_csv(RAW / "departments.csv")
aisles = pd.read_csv(RAW / "aisles.csv")
products = pd.read_csv(RAW / "products.csv")
orders = pd.read_csv(RAW / "orders.csv")

for name, df in [("departments", departments), ("aisles", aisles), ("products", products), ("orders", orders)]:
    print(name, df.shape)
display(products.head())
display(orders.head())
display(orders.isna().mean().to_frame("null_share"))

In [ ]:
eval_counts = orders.groupby("eval_set").agg(orders=("order_id", "size"), users=("user_id", "nunique"))
display(eval_counts)

orders_per_user = orders.groupby("user_id")["order_number"].max()
print("Users:", len(orders_per_user))
print("Orders per user: median", orders_per_user.median(), " mean", round(orders_per_user.mean(), 1), " max", orders_per_user.max())

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
eval_counts["orders"].plot.bar(ax=ax[0], color="#4c72b0", rot=0, title="Orders by eval_set")
orders_per_user.plot.hist(ax=ax[1], bins=50, color="#4c72b0", title="Orders per user")
plt.tight_layout()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
orders.groupby("order_dow").size().plot.bar(ax=ax[0], color="#4c72b0", rot=0, title="Orders by day of week (0-6)")
orders.groupby("order_hour_of_day").size().plot.bar(ax=ax[1], color="#4c72b0", title="Orders by hour of day")
plt.tight_layout()

heat = orders.pivot_table(index="order_dow", columns="order_hour_of_day", values="order_id", aggfunc="count")
plt.figure(figsize=(12, 3.5))
plt.imshow(heat, aspect="auto", cmap="Blues")
plt.colorbar(label="orders"); plt.xlabel("hour of day"); plt.ylabel("day of week"); plt.title("Orders: day of week x hour")
plt.grid(False)

In [ ]:
orders["days_since_prior_order"].plot.hist(bins=31, color="#55a868", title="Days since prior order (30 = capped)")
print("Share of orders at the 30-day cap:", f"{(orders['days_since_prior_order'] == 30).mean():.1%}")

In [ ]:
prior = pd.read_csv(RAW / "order_products__prior.csv", dtype="int32")
last = pd.read_csv(RAW / "order_products__train.csv", dtype="int32")
print("prior lines:", f"{len(prior):,}", " last-order lines:", f"{len(last):,}")

basket = prior.groupby("order_id").size()
print("Basket size: mean", round(basket.mean(), 1), " median", basket.median(), " p99", basket.quantile(.99))
basket.clip(upper=60).plot.hist(bins=60, color="#c44e52", title="Basket size (products per order, capped at 60)")

In [ ]:
print("Overall reorder rate:", f"{prior['reordered'].mean():.1%}")

pos = prior[prior["add_to_cart_order"] <= 40].groupby("add_to_cart_order")["reordered"].mean()
pos.plot(marker="o", color="#c44e52", title="Reorder rate by position in cart")
plt.xlabel("add_to_cart_order")

In [ ]:
per_product = prior.groupby("product_id").size().rename("times_ordered")
catalog = products.merge(departments).merge(aisles).merge(per_product, left_on="product_id", right_index=True, how="left").fillna({"times_ordered": 0})

top = catalog.sort_values("times_ordered", ascending=False).head(20)
top.set_index("product_name")["times_ordered"].plot.barh(figsize=(8, 6), color="#4c72b0", title="Top-20 products").invert_yaxis()

In [ ]:
dep = catalog.groupby("department").agg(products=("product_id", "size"), times_ordered=("times_ordered", "sum"))
dep["order_share"] = dep["times_ordered"] / dep["times_ordered"].sum()

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
dep["products"].sort_values().plot.barh(ax=ax[0], color="#8172b2", title="Products per department")
dep["order_share"].sort_values().plot.barh(ax=ax[1], color="#8172b2", title="Share of ordered items by department")
plt.tight_layout()

In [ ]:
merged = prior[["product_id", "reordered"]].merge(products[["product_id", "department_id"]]).merge(departments)
dep_reorder = merged.groupby("department")["reordered"].mean().sort_values()
dep_reorder.plot.barh(figsize=(8, 5), color="#55a868", title="Reorder rate by department")
del merged

In [ ]:
ranked = per_product.sort_values(ascending=False)
share = ranked.cumsum() / ranked.sum()
x = np.arange(1, len(share) + 1) / len(share)
n80 = (share < .8).sum() / len(share)
plt.plot(x, share.values)
plt.axhline(.8, ls="--", c="gray"); plt.axvline(n80, ls="--", c="gray")
plt.title(f"{n80:.1%} of products make 80% of ordered items")
plt.xlabel("share of products (most popular first)"); plt.ylabel("share of ordered items")
print("Products never ordered in prior:", int((catalog['times_ordered'] == 0).sum()))

In [ ]:
user_of = np.zeros(orders["order_id"].max() + 1, dtype="int32")
user_of[orders["order_id"].to_numpy()] = orders["user_id"].to_numpy()

prior_keys = np.unique(user_of[prior["order_id"].to_numpy()].astype("int64") * 100_000 + prior["product_id"].to_numpy())
n_users, n_products = orders["user_id"].nunique(), len(products)
print(f"User-product pairs: {len(prior_keys):,}")
print(f"Matrix density: {len(prior_keys) / (n_users * n_products):.4%}")

products_per_user = pd.Series(prior_keys // 100_000).value_counts()
products_per_user.plot.hist(bins=60, color="#4c72b0", title="Distinct products per user")

In [ ]:
last_keys = user_of[last["order_id"].to_numpy()].astype("int64") * 100_000 + last["product_id"].to_numpy()
seen_before = np.isin(last_keys, prior_keys)
print("Items in the last order that the user had already bought before:", f"{seen_before.mean():.1%}")
print("Last-order items that are new for the user:", f"{(~seen_before).mean():.1%}")

last_basket = last.groupby("order_id").size()
print("Last-order basket size: mean", round(last_basket.mean(), 1))
pd.Series({"already bought": seen_before.mean(), "new for user": 1 - seen_before.mean()}).plot.bar(color=["#55a868", "#c44e52"], rot=0, title="Last-order items: repeat vs new")